<a href="https://colab.research.google.com/github/GGSimmons1992/UTYV6k8pXAuLL0yL/blob/main/Notebooks/DataPrep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1a: Classical Random Forest Data Prep

In [1]:
!pip install category_encoders

In [2]:
import numpy as np
import numpy
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as rf
import pickle
from imblearn.over_sampling import RandomOverSampler, SMOTE, SMOTENC
from os.path import exists
from sklearn.preprocessing import StandardScaler
import category_encoders as ce
from sklearn.feature_selection import chi2
from scipy.stats import spearmanr
import json
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def fileExistsInDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  return exists(fullName)

In [4]:
def savePickleFileToDrive(data,filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  pickle.dump(data, open(fullName, 'wb'))

In [5]:
def retrievePickleFileFromDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  return pickle.load(open(fullName, 'rb'))

In [6]:
def saveJSONToDrive(data,filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  with open(fullName, 'w') as fp:
      json.dump(data, fp)

In [7]:
def retrieveJSONFromDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  with open(fullName) as json_file:
      data = json.load(json_file)
  return data

In [8]:
def saveCSVToDrive(df,filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  df.to_csv(fullName,index=False)

In [9]:
def retrieveCSVFromDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  return pd.read_csv(fullName)

In [10]:
def separateDFBySubtype(df,baseName):
  numericalCols = []
  categoricalCols = []
  for col in df.columns:
      if np.issubdtype(df[col].dtype, np.number):
          numericalCols.append(str(col))
      else:
          categoricalCols.append(str(col))
  numericalDF = df[numericalCols]
  categoricalDF = df[categoricalCols]
  savePickleFileToDrive(numericalCols, f"{baseName}NumericalCols.pkl")
  savePickleFileToDrive(categoricalCols, f"{baseName}CategoricalCols.pkl")
  return numericalDF,categoricalDF

In [11]:
def encodeTestDF(categoricalDF,baseName):
  ohe = retrievePickleFileFromDrive(f"{baseName}OneHotEncoder.pkl")
  le = retrievePickleFileFromDrive(f"{baseName}LabelEncoder.pkl")
  oheDF = ohe.transform(categoricalDF).fillna(0)
  leDF = le.transform(categoricalDF)
  return pd.concat([oheDF,leDF],axis=1)

In [12]:
def encodeDF(categoricalDF,baseName):
  ohe = ce.OneHotEncoder(handle_unknown='return_nan',return_df=True,use_cat_names=True)
  le = ce.OrdinalEncoder(return_df=True)
  oheDF = ohe.fit_transform(categoricalDF)
  oheColumns = list(oheDF.columns)
  savePickleFileToDrive(oheColumns,f"{baseName}OheColumns.pkl")
  leDF = le.fit_transform(categoricalDF)
  savePickleFileToDrive(ohe,f"{baseName}OneHotEncoder.pkl")
  savePickleFileToDrive(le,f"{baseName}LabelEncoder.pkl")
  return pd.concat([oheDF,leDF],axis=1)

In [13]:
def scaleTestDF(df,baseName):
  scaler = retrievePickleFileFromDrive(f"{baseName}Scaler.pkl")
  numericalCols = list(df.columns)
  # Use .loc for explicit assignment to prevent SettingWithCopyWarning
  df.loc[:, numericalCols] = scaler.transform(df.loc[:, numericalCols])
  return df

In [14]:
def scaleDF(df,baseName):
  scaler = StandardScaler()
  numericalCols = list(df.columns)
  # Use .loc for explicit assignment to prevent SettingWithCopyWarning
  df.loc[:, numericalCols] = scaler.fit_transform(df.loc[:, numericalCols])
  savePickleFileToDrive(scaler,f"{baseName}Scaler.pkl")
  return df

In [15]:
def processTestData(baseName, splitName='Test'):
  # Process any non-training split (like 'Test' or 'Dev') using fitted parameters
  df = retrieveCSVFromDrive(f"{baseName}{splitName}.csv")
  yArray = df[['isSubscribed']].values.ravel()
  df = df.drop('isSubscribed', axis=1)

  rawDF = df.copy()
  saveCSVToDrive(rawDF, f"{baseName}Raw{splitName}.csv")

  numericalDF, categoricalDF = separateDFBySubtype(df, baseName)
  scaledDF = scaleTestDF(numericalDF, baseName)
  encodedDF = encodeTestDF(categoricalDF, baseName)
  finalDF = pd.concat([scaledDF, encodedDF], axis=1)
  finalDF['isSubscribed'] = yArray.reshape(-1, 1)
  saveCSVToDrive(finalDF, f"{baseName}{splitName}.csv")

In [16]:
def processTrainData(baseName):
  df = retrieveCSVFromDrive(f"{baseName}Train.csv")

  yArray = df[['isSubscribed']].values.ravel()
  df = df.drop('isSubscribed',axis = 1)


  rawDF = df.copy()
  saveCSVToDrive(rawDF,f"{baseName}RawTrain.csv")

  numericalDF,categoricalDF = separateDFBySubtype(df,baseName)
  scaledDF = scaleDF(numericalDF,baseName)
  encodedDF = encodeDF(categoricalDF,baseName)
  finalDF = pd.concat([scaledDF,encodedDF],axis=1)
  finalDF['isSubscribed'] = yArray.reshape(-1,1)
  saveCSVToDrive(finalDF,f"{baseName}Train.csv")


In [17]:
def transformDate(data):
  dateTimeColumns = ['First Contact','Last Contact', 'First Call', 'Signed up for a demo',
                     'Filled in customer survey','Did sign up to the platform','Account Manager assigned']
  for col in dateTimeColumns:
    data[col] = pd.to_datetime(data[col], format='%Y-%d-%m', errors='coerce')
    data[col + 'Year'] = data[col].dt.year
    data[col + 'Month'] = data[col].dt.month
    data[col + 'Day'] = data[col].dt.day
    data[col] = data[col].astype('int64')
  return data

In [18]:
def binarizeTargetsFromDF(df):
    df['Subscribed'] = pd.to_datetime(df['Subscribed'], format='%Y-%d-%m', errors='coerce').astype('int64')
    df['isSubscribed'] = (df['Subscribed'] > 0).astype(int)
    df.drop('Subscribed',axis=1,inplace=True)
    return df

In [19]:
def overSample(X, y, baseName):
    print("[Oversampling] Applying SMOTENC to the training subset...")
    # Identify the index of columns that are categorical (not numeric)
    categorical_features_indices = [i for i, col in enumerate(X.columns) if not np.issubdtype(X[col].dtype, np.number)]

    # Set k_neighbors=2 because the minority class contains only 5 samples in the train split
    balancer = SMOTENC(categorical_features=categorical_features_indices, k_neighbors=2, random_state=51)
    resampledX, resampledy = balancer.fit_resample(X, y)
    return resampledX, resampledy

In [20]:
def splitData(baseName):
    df = retrieveCSVFromDrive("cleanedData.csv")
    df = binarizeTargetsFromDF(df)
    originalY = df[['isSubscribed']]
    originalYArray = originalY.values.ravel()
    originalX = df.drop('isSubscribed', axis=1)

    # 1. Split into temporary train and final test sets (60/40 instead of 80/20 to keep more in evaluation sets)
    XTempTrain, XTest, yTempTrain, yTest = train_test_split(
        originalX, originalYArray, test_size=0.4, random_state=51, stratify=originalYArray
    )

    # 2. Split temporary train further into Sub-Train and Dev/Validation sets (60/40)
    XSubTrain, XDev, ySubTrain, yDev = train_test_split(
        XTempTrain, yTempTrain, test_size=0.4, random_state=51, stratify=yTempTrain
    )

    # 3. Apply oversampling ONLY to the Sub-Train data
    resampledXSubTrain, resampledYSubTrain = overSample(XSubTrain, ySubTrain, baseName)

    # 4. Save the three datasets
    # Save Train (Oversampled Sub-Train)
    train = resampledXSubTrain.copy()
    train['isSubscribed'] = resampledYSubTrain
    saveCSVToDrive(train, f"{baseName}Train.csv")

    # Save Dev (Untouched Dev Set)
    dev = XDev.copy()
    dev['isSubscribed'] = yDev
    saveCSVToDrive(dev, f"{baseName}Dev.csv")

    # Save Test (Untouched Test Set)
    test = XTest.copy()
    test['isSubscribed'] = yTest
    saveCSVToDrive(test, f"{baseName}Test.csv")

In [21]:
def replaceNanWithZero(df):
  '''
  For Country and Education, sure it means we didn't get that info in, but for all the other columns,
  this is a status or date. No status or date, means said action never happened.
  e.g. person wasn't contacted, person did not subscribe, person wasn't assigned a manager,
  person is not in a stage or status etc...
  '''
  for col in df.columns:
    df[col] = df[col].fillna(0)
  return df

In [22]:
def removeStage(df):
  df.drop('Stage',axis=1,inplace=True)
  return df

In [23]:
def main():
  data = retrieveCSVFromDrive("SalesCRM - CRM.csv")
  dateTransformedData = transformDate(data)
  stageLessData = removeStage(dateTransformedData)
  nonNanData = replaceNanWithZero(stageLessData)
  baseName = "SalesReinforcer"
  saveCSVToDrive(nonNanData, "cleanedData.csv")
  np.random.seed(51)
  # Force re-splitting to apply updated split configuration
  splitData(baseName)
  processTrainData(baseName)
  processTestData(baseName, splitName='Dev')
  processTestData(baseName, splitName='Test')

In [24]:
if __name__ == "__main__":
    main()

    # Diagnostic cell to check datasets
    train_df = retrieveCSVFromDrive("SalesReinforcerTrain.csv")
    dev_df = retrieveCSVFromDrive("SalesReinforcerDev.csv")
    test_df = retrieveCSVFromDrive("SalesReinforcerTest.csv")

    print("Train shape:", train_df.shape)
    print("Dev shape:", dev_df.shape)
    print("Test shape:", test_df.shape)
    print("\nTrain target class distribution:\n", train_df['isSubscribed'].value_counts())
    print("\nDev target class distribution:\n", dev_df['isSubscribed'].value_counts())
    print("\nTest target class distribution:\n", test_df['isSubscribed'].value_counts())

[Oversampling] Applying SMOTENC to the training subset...


/tmp/ipykernel_3358/3658709943.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 2.06071317  2.11892446  0.24067578 ... -0.4095283  -0.79858265
 -0.84504488]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, numericalCols] = scaler.fit_transform(df.loc[:, numericalCols])
/tmp/ipykernel_3358/3658709943.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.61368559  1.90138289 -0.61368559 ...  1.62788097 -0.61368559
 -0.61368559]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, numericalCols] = scaler.fit_transform(df.loc[:, numericalCols])
/tmp/ipykernel_3358/3658709943.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.61413792  1.89949472 -0.61413792 ...  1.62832

Train shape: (7932, 141)
Dev shape: (2648, 141)
Test shape: (4413, 141)

Train target class distribution:
 isSubscribed
0    3966
1    3966
Name: count, dtype: int64

Dev target class distribution:
 isSubscribed
0    2645
1       3
Name: count, dtype: int64

Test target class distribution:
 isSubscribed
0    4408
1       5
Name: count, dtype: int64
